# 01 – Data acquisition and input checks

**Project:** Premature termination of Phase I trials in Europe
**Purpose of this notebook:** download the raw data from the three registries and check
that they contain what we expect, before any cleaning happens.

| Part | Registry | Block |
|---|---|---|
| A | ClinicalTrials.gov (API v2) – main dataset | B2 |
| B | EU Clinical Trials Register (EudraCT) | B3 |
| C | CTIS (euclinicaltrials.eu) | B4 |

No filtering by status or start year happens here – that is done in notebook 02, so every
exclusion step stays traceable.

In [1]:
# --- Setup -------------------------------------------------------------------
import gzip
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

import offline_guard

OFFLINE = offline_guard.activate()   # PHASE1_OFFLINE=1 → re-run from saved raw data only
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Project root = parent of the notebooks/ folder
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PATHS = {
    "raw_ctgov": PROJECT_DIR / "data" / "raw" / "ctgov",
    "raw_eudract": PROJECT_DIR / "data" / "raw" / "eudract",
    "raw_ctis": PROJECT_DIR / "data" / "raw" / "ctis",
    "interim": PROJECT_DIR / "data" / "interim",
}
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.name)   # folder name only (no user path in outputs)

OFFLINE run: network access blocked – only saved raw data are used.
Project directory: phase1-terminations-europe


## Part A – ClinicalTrials.gov

### A1. Query definition

* **Study type:** interventional
* **Phase:** Phase 1, including Phase 1/2 (`AREA[Phase]PHASE1` matches any trial whose phase
  list contains PHASE1). Early Phase 1 is a separate value and is checked below.
* **Location:** at least one site in one of the 7 countries (`query.locn`). This is a text
  search, so in A4 we verify against the structured site countries.
* **Fields:** protocol section modules + MeSH browse modules + `hasResults`. The full
  results section is not downloaded (large, not needed).

In [2]:
API_BASE = "https://clinicaltrials.gov/api/v2"

TARGET_COUNTRIES = ["United Kingdom", "Germany", "France", "Spain",
                    "Netherlands", "Belgium", "Italy"]

QUERY_PARAMS = {
    "query.locn": " OR ".join(f'"{c}"' for c in TARGET_COUNTRIES),
    "filter.advanced": "AREA[StudyType]INTERVENTIONAL AND AREA[Phase]PHASE1",
}

FIELDS = [
    "protocolSection.identificationModule",
    "protocolSection.statusModule",
    "protocolSection.sponsorCollaboratorsModule",
    "protocolSection.oversightModule",
    "protocolSection.conditionsModule",
    "protocolSection.designModule",
    "protocolSection.armsInterventionsModule",
    "protocolSection.eligibilityModule",
    "protocolSection.contactsLocationsModule",
    "derivedSection.conditionBrowseModule",
    "derivedSection.interventionBrowseModule",
    "hasResults",
]

RAW_FILE = PATHS["raw_ctgov"] / "ctgov_phase1_europe_raw.jsonl.gz"
META_FILE = PATHS["raw_ctgov"] / "ctgov_phase1_europe_raw_metadata.json"

# Set to True to download again (otherwise the saved raw file is reused)
FORCE_DOWNLOAD = False

### A2. Download (with pagination)

The API returns at most 1,000 studies per page plus a `nextPageToken` for the next page.
The raw records are saved unchanged as compressed JSON Lines (one study per line),
together with a metadata file (query, API version, data timestamp, download time).

In [3]:
def download_ctgov(params, fields, page_size=1000):
    """Download all studies matching the query; returns (studies, total_count)."""
    studies, token, total = [], None, None
    while True:
        p = {**params, "fields": ",".join(fields), "pageSize": page_size, "countTotal": "true"}
        if token:
            p["pageToken"] = token
        r = requests.get(f"{API_BASE}/studies", params=p, timeout=120)
        r.raise_for_status()
        data = r.json()
        total = data.get("totalCount", total)
        studies.extend(data["studies"])
        print(f"  downloaded {len(studies):>6,} / {total:,}")
        token = data.get("nextPageToken")
        if not token:
            return studies, total


if RAW_FILE.exists() and not FORCE_DOWNLOAD:
    with gzip.open(RAW_FILE, "rt", encoding="utf-8") as f:
        studies = [json.loads(line) for line in f]
    meta = json.loads(META_FILE.read_text())
    print(f"Loaded {len(studies):,} studies from saved raw file (downloaded {meta['downloaded_at_utc']})")
else:
    version = requests.get(f"{API_BASE}/version", timeout=30).json()
    studies, total_count = download_ctgov(QUERY_PARAMS, FIELDS)
    with gzip.open(RAW_FILE, "wt", encoding="utf-8") as f:
        for s in studies:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    meta = {
        "source": "ClinicalTrials.gov API v2",
        "endpoint": f"{API_BASE}/studies",
        "query_params": QUERY_PARAMS,
        "fields": FIELDS,
        "api_version": version["apiVersion"],
        "data_timestamp": version["dataTimestamp"],
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "total_count_reported": total_count,
        "n_records_saved": len(studies),
    }
    META_FILE.write_text(json.dumps(meta, indent=2))

print(json.dumps({k: meta[k] for k in ["api_version", "data_timestamp", "downloaded_at_utc",
                                       "total_count_reported", "n_records_saved"]}, indent=2))
print(f"Raw file size: {RAW_FILE.stat().st_size / 1e6:.1f} MB")

Loaded 11,120 studies from saved raw file (downloaded 2026-10-27T20:13:22+00:00)
{
  "api_version": "2.0.5",
  "data_timestamp": "2026-10-27T09:00:04",
  "downloaded_at_utc": "2026-10-27T20:13:22+00:00",
  "total_count_reported": 11120,
  "n_records_saved": 11120
}
Raw file size: 23.3 MB


### A3. Flatten to one row per trial

Nested JSON is turned into a flat table with the fields we need later. Lists (e.g. phases,
countries, MeSH terms) are stored as `|`-separated text. Dates stay as text here; they are
parsed in notebook 02. EudraCT numbers (format `YYYY-NNNNNN-NN`) are extracted from the
study IDs for the registry cross-check in Part B.

In [4]:
EUDRACT_PATTERN = re.compile(r"\b(?:19|20)\d{2}-\d{6}-\d{2}\b")


def get(d, *keys, default=None):
    """Safely walk nested dictionaries: get(d, 'a', 'b') == d['a']['b'] or default."""
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d


def join(values):
    """Join a list into '|'-separated text (None if empty)."""
    values = [str(v) for v in values if v not in (None, "")]
    return "|".join(values) if values else None


def flatten_study(s):
    ps = s.get("protocolSection", {})
    ident, status = ps.get("identificationModule", {}), ps.get("statusModule", {})
    sponsor, design = ps.get("sponsorCollaboratorsModule", {}), ps.get("designModule", {})
    arms, elig = ps.get("armsInterventionsModule", {}), ps.get("eligibilityModule", {})
    locations = get(ps, "contactsLocationsModule", "locations", default=[])
    cond_browse = get(s, "derivedSection", "conditionBrowseModule", default={})

    countries = sorted({loc.get("country") for loc in locations if loc.get("country")})
    target = [c for c in countries if c in TARGET_COUNTRIES]
    secondary_ids = [x.get("id") for x in ident.get("secondaryIdInfos", [])]
    eudract_ids = sorted(set(EUDRACT_PATTERN.findall(" ".join(
        [ident.get("orgStudyId") or ""] + [i or "" for i in secondary_ids]))))

    return {
        "nct_id": ident.get("nctId"),
        "brief_title": ident.get("briefTitle"),
        "org_study_id": ident.get("orgStudyId"),
        "secondary_ids": join(secondary_ids),
        "eudract_ids": join(eudract_ids),
        # status and dates
        "overall_status": status.get("overallStatus"),
        "why_stopped": status.get("whyStopped"),
        "status_verified_date": status.get("statusVerifiedDate"),
        "start_date": get(status, "startDateStruct", "date"),
        "start_date_type": get(status, "startDateStruct", "type"),
        "primary_completion_date": get(status, "primaryCompletionDateStruct", "date"),
        "completion_date": get(status, "completionDateStruct", "date"),
        "first_submit_date": status.get("studyFirstSubmitDate"),
        "first_post_date": get(status, "studyFirstPostDateStruct", "date"),
        "last_update_post_date": get(status, "lastUpdatePostDateStruct", "date"),
        # sponsor
        "lead_sponsor_name": get(sponsor, "leadSponsor", "name"),
        "lead_sponsor_class": get(sponsor, "leadSponsor", "class"),
        "n_collaborators": len(sponsor.get("collaborators", [])),
        "collaborator_classes": join(sorted({c.get("class") for c in sponsor.get("collaborators", [])})),
        "has_dmc": get(ps, "oversightModule", "oversightHasDmc"),
        # condition
        "conditions": join(get(ps, "conditionsModule", "conditions", default=[])),
        "condition_mesh_terms": join([m.get("term") for m in cond_browse.get("meshes", [])]),
        "condition_mesh_ancestors": join([m.get("term") for m in cond_browse.get("ancestors", [])]),
        # design
        "study_type": design.get("studyType"),
        "phases": join(design.get("phases", [])),
        "allocation": get(design, "designInfo", "allocation"),
        "intervention_model": get(design, "designInfo", "interventionModel"),
        "primary_purpose": get(design, "designInfo", "primaryPurpose"),
        "masking": get(design, "designInfo", "maskingInfo", "masking"),
        "enrollment_count": get(design, "enrollmentInfo", "count"),
        "enrollment_type": get(design, "enrollmentInfo", "type"),
        "n_arm_groups": len(arms.get("armGroups", [])),
        "arm_group_types": join(sorted({a.get("type") for a in arms.get("armGroups", [])})),
        "intervention_types": join(sorted({i.get("type") for i in arms.get("interventions", [])})),
        # population
        "healthy_volunteers": elig.get("healthyVolunteers"),
        "sex": elig.get("sex"),
        "std_ages": join(elig.get("stdAges", [])),
        # locations
        "n_locations": len(locations),
        "location_countries": join(countries),
        "n_countries": len(countries),
        "target_countries": join(target),
        "n_target_countries": len(target),
        # results
        "has_results": s.get("hasResults"),
    }


ctgov = pd.DataFrame([flatten_study(s) for s in studies])
print(ctgov.shape)
ctgov.head(3)

(11120, 43)


,nct_id,brief_title,org_study_id,secondary_ids,eudract_ids,overall_status,why_stopped,status_verified_date,start_date,start_date_type,primary_completion_date,completion_date,first_submit_date,first_post_date,last_update_post_date,lead_sponsor_name,lead_sponsor_class,n_collaborators,collaborator_classes,has_dmc,conditions,condition_mesh_terms,condition_mesh_ancestors,study_type,phases,allocation,intervention_model,primary_purpose,masking,enrollment_count,enrollment_type,n_arm_groups,arm_group_types,intervention_types,healthy_volunteers,sex,std_ages,n_locations,location_countries,n_countries,target_countries,n_target_countries,has_results
0,NCT01601106,Silencing Inflammatory Activity by Injecting N...,None,NaN,NaN,UNKNOWN,NaN,2012-05,2011-09,NaN,2013-05,NaN,2011-10-13,2012-05-17,2012-05-21,Academisch Medisch Centrum - Universiteit van ...,OTHER,0,NaN,None,Atherosclerosis|Inflammation,Atherosclerosis|Inflammation,Arteriosclerosis|Arterial Occlusive Diseases|V...,INTERVENTIONAL,PHASE1|PHASE2,RANDOMIZED,PARALLEL,TREATMENT,QUADRUPLE,30.0,ESTIMATED,2,ACTIVE_COMPARATOR|PLACEBO_COMPARATOR,DRUG,False,ALL,ADULT|OLDER_ADULT,1,Netherlands,1,Netherlands,1,False
1,NCT04978506,A Study in Healthy Men to Test How Well Differ...,None,2021-001717-36,2021-001717-36,COMPLETED,NaN,2026-06,2021-09-03,ACTUAL,2024-07-04,2024-07-04,2021-07-26,2021-07-27,2026-06-26,Boehringer Ingelheim,INDUSTRY,0,NaN,False,Healthy,NaN,NaN,INTERVENTIONAL,PHASE1,RANDOMIZED,PARALLEL,TREATMENT,SINGLE,83.0,ACTUAL,9,EXPERIMENTAL|PLACEBO_COMPARATOR,DRUG,True,MALE,ADULT|OLDER_ADULT,1,Germany,1,Germany,1,True
2,NCT01545947,"Study Assessing Safety, Pharmacokinetics and E...",None,2011-005290-23,2011-005290-23,COMPLETED,NaN,2019-11,2012-05-01,ACTUAL,2014-12-11,2014-12-11,2012-03-02,2012-03-07,2019-11-12,Celgene,INDUSTRY,0,NaN,False,"Carcinoma, Non-Small-Cell Lung|Non-Small Cell ...","Carcinoma, Non-Small-Cell Lung|Neoplasms|Lung ...","Carcinoma, Bronchogenic|Bronchial Neoplasms|Re...",INTERVENTIONAL,PHASE1,NON_RANDOMIZED,PARALLEL,TREATMENT,NONE,76.0,ACTUAL,3,EXPERIMENTAL,DRUG,False,ALL,ADULT|OLDER_ADULT,9,Spain|United States,2,Spain,1,False


### A4. Input checks

#### A4.1 Completeness of the download and duplicates

In [5]:
n_dupes = ctgov["nct_id"].duplicated().sum()
print(f"Records reported by API : {meta['total_count_reported']:,}")
print(f"Records downloaded      : {len(ctgov):,}")
print(f"Duplicate NCT IDs       : {n_dupes}")
assert len(ctgov) == meta["total_count_reported"], "Download incomplete"
assert n_dupes == 0, "Duplicate NCT IDs found"

Records reported by API : 11,120
Records downloaded      : 11,120
Duplicate NCT IDs       : 0


#### A4.2 Study type and phases

In [6]:
print(ctgov["study_type"].value_counts(dropna=False), "\n")
print(ctgov["phases"].value_counts(dropna=False))

has_phase1 = ctgov["phases"].fillna("").str.split("|").apply(lambda p: "PHASE1" in p)
has_early = ctgov["phases"].fillna("").str.contains("EARLY_PHASE1")
print(f"\nTrials without PHASE1 in phase list : {(~has_phase1).sum()}")
print(f"Trials with EARLY_PHASE1            : {has_early.sum()}")

study_type
INTERVENTIONAL    11120
Name: count, dtype: int64 

phases
PHASE1           8069
PHASE1|PHASE2    3051
Name: count, dtype: int64

Trials without PHASE1 in phase list : 0
Trials with EARLY_PHASE1            : 0


#### A4.3 Countries

The location search is a text search. Here we check against the structured country of each
site. Trials without any site in the 7 target countries are flagged
(`in_target_countries = False`) and will be excluded in notebook 02.

In [7]:
ctgov["in_target_countries"] = ctgov["n_target_countries"] > 0
no_target = ctgov[~ctgov["in_target_countries"]]
print(f"Trials with >=1 site in a target country : {ctgov['in_target_countries'].sum():,}")
print(f"Trials without (text match only)         : {len(no_target):,}")
print(f"  ... of which with no site list at all  : {(no_target['n_locations'] == 0).sum():,}")
no_target[["nct_id", "brief_title", "n_locations", "location_countries"]].head(10)

Trials with >=1 site in a target country : 11,114
Trials without (text match only)         : 6
  ... of which with no site list at all  : 0


,nct_id,brief_title,n_locations,location_countries
1198,NCT06006299,Investigating the Use of taVNS to Treat Insomn...,2,United States
2789,NCT00333424,Thai Prophylactic HIV Vaccine Phase I Study,1,Thailand
7992,NCT05585307,Study of Novel Antiretrovirals in Participants...,42,Dominican Republic|Thailand|United States
8246,NCT00477126,Bioequivalence Study of Generic GPO Ritonavir ...,1,Thailand
8317,NCT00851435,Safety and Pharmacokinetics of KBPA-101 in Hos...,1,Switzerland
8792,NCT07001319,Study of GS-3242 in Participants With HIV-1; S...,28,Mexico|Thailand|United States


Comparison with the manual CSV exports from ClinicalTrials.gov (one file per country, all
years and statuses, downloaded earlier). Small differences are expected because the registry
is updated daily and the CSV search also matched text.

In [8]:
CSV_REFERENCE = {"United Kingdom": 3926, "Germany": 3154, "France": 2435, "Spain": 2403,
                 "Netherlands": 1697, "Belgium": 1646, "Italy": 1303}

per_country = (ctgov["target_countries"].dropna().str.split("|").explode()
               .value_counts().rename("api_trials_with_site"))
country_check = pd.DataFrame({"csv_export": pd.Series(CSV_REFERENCE)}).join(per_country)
country_check["difference"] = country_check["api_trials_with_site"] - country_check["csv_export"]
country_check["difference_pct"] = (100 * country_check["difference"] / country_check["csv_export"]).round(1)
print(f"Sum over countries (multi-country trials counted repeatedly): "
      f"{country_check['api_trials_with_site'].sum():,}")
print(f"Unique trials                                               : "
      f"{ctgov['in_target_countries'].sum():,}\n")
country_check

Sum over countries (multi-country trials counted repeatedly): 16,561
Unique trials                                               : 11,114



,csv_export,api_trials_with_site,difference,difference_pct
United Kingdom,3926,3927,1,0.0
Germany,3154,3157,3,0.1
France,2435,2433,-2,-0.1
Spain,2403,2402,-1,-0.0
Netherlands,1697,1692,-5,-0.3
Belgium,1646,1646,0,0.0
Italy,1303,1304,1,0.1


In [9]:
# How many target countries per trial?
ctgov.loc[ctgov["in_target_countries"], "n_target_countries"].value_counts().sort_index()

n_target_countries
1    8552
2    1079
3     656
4     423
5     267
6     103
7      34
Name: count, dtype: int64

#### A4.4 Overall status and start year (all trials, before filtering)

In [10]:
print(ctgov["overall_status"].value_counts(dropna=False))

overall_status
COMPLETED                  7610
TERMINATED                 1279
RECRUITING                  909
ACTIVE_NOT_RECRUITING       544
UNKNOWN                     533
WITHDRAWN                   134
NOT_YET_RECRUITING           71
SUSPENDED                    28
ENROLLING_BY_INVITATION      12
Name: count, dtype: int64


In [11]:
start_year = pd.to_numeric(ctgov["start_date"].str[:4], errors="coerce")
year_bins = pd.cut(start_year, bins=[1900, 2004, 2015, 2016, 2022, 2100],
                   labels=["<2005", "2005-2015", "2016", "2017-2022", ">=2023"])
print(year_bins.value_counts(dropna=False).sort_index())
print(f"\nMissing start date: {start_year.isna().sum()}")

start_date
<2005         331
2005-2015    4975
2016          578
2017-2022    3527
>=2023       1676
NaN            33
Name: count, dtype: int64

Missing start date: 33


#### A4.5 Completeness of key fields

Share of trials with a non-missing value. For `why_stopped`, only terminated trials are
relevant.

In [12]:
KEY_FIELDS = ["overall_status", "start_date", "lead_sponsor_class", "healthy_volunteers",
              "enrollment_count", "allocation", "masking", "primary_purpose", "arm_group_types",
              "conditions", "condition_mesh_terms", "location_countries", "has_results"]
completeness = (ctgov[KEY_FIELDS].notna().mean() * 100).round(1).rename("% non-missing").to_frame()

terminated = ctgov["overall_status"] == "TERMINATED"
why_ok = ctgov.loc[terminated, "why_stopped"].notna().mean() * 100
completeness.loc["why_stopped (terminated only)"] = round(why_ok, 1)
completeness

,% non-missing
overall_status,100.0
start_date,99.7
lead_sponsor_class,100.0
healthy_volunteers,99.8
enrollment_count,99.6
allocation,98.6
masking,99.0
primary_purpose,96.8
arm_group_types,95.4
conditions,100.0


In [13]:
# Value distributions of the main categorical fields
for col in ["lead_sponsor_class", "healthy_volunteers", "allocation", "masking", "enrollment_type"]:
    print(ctgov[col].value_counts(dropna=False).to_string(), "\n")

lead_sponsor_class
INDUSTRY     8347
OTHER        2624
NETWORK        70
OTHER_GOV      66
NIH             6
UNKNOWN         4
INDIV           3 

healthy_volunteers
False    6608
True     4495
None       17 

allocation
RANDOMIZED        5189
NON_RANDOMIZED    2945
NA                2831
NaN                155 

masking
NONE         7634
DOUBLE       1388
QUADRUPLE     960
TRIPLE        570
SINGLE        458
NaN           110 

enrollment_type
ACTUAL       8952
ESTIMATED    1993
NaN           175 



In [14]:
# A first look at termination reasons (cleaned and classified in notebook 03)
ctgov.loc[terminated, "why_stopped"].sample(10, random_state=42).tolist()

["The decision to discontinue the trial was made as part of a strategic review of the Sponsor's global development portfolio. It is not related to any safety concerns, efficacy findings, recruitment difficulties, or issues with trial conduct.",
 'Freeline has decided to pause development of FLT180a for business priorities',
 'A decision to stop the study was made by the site and GSK project team based on pre-defined stopping criteria in the protocol',
 'This trial was supporting the development of an asset which was terminated, thus the data from the study were no longer required.',
 'This trial was concluded for strategic reasons.',
 'poor tolerability at highest dose',
 'Sponsor decision',
 'This was a strategic business decision. There were no safety concerns contributing to this decision.',
 'Drug toxicity',
 'Business objectives have changed']

### A5. Save the flat table

Saved to `data/interim/` (not published). Notebook 02 starts from this file.

In [15]:
FLAT_FILE = PATHS["interim"] / "ctgov_flat.parquet"
ctgov.to_parquet(FLAT_FILE, index=False)
print(f"Saved {len(ctgov):,} rows x {ctgov.shape[1]} columns to {FLAT_FILE.relative_to(PROJECT_DIR)}")

Saved 11,120 rows x 44 columns to data/interim/ctgov_flat.parquet


### A6. Summary of the ClinicalTrials.gov input checks

Data snapshot: API 2.0.5, data timestamp 2026-10-27.

| Check | Result |
|---|---|
| Records downloaded vs. reported | 11,120 / 11,120 – complete |
| Duplicate NCT IDs | 0 (one query across all countries) |
| Study type | 100 % interventional |
| Phases | Phase 1: 8,069 · Phase 1/2: 3,051 · Early Phase 1: 0 · other: 0 |
| Site in a target country | 11,114 (6 text-only matches flagged for exclusion) |
| Per-country counts vs. manual CSV exports | deviation ≤ 0.3 % per country |
| Multi-country trials | 23 % of trials have sites in ≥ 2 target countries |
| `why_stopped` filled (terminated trials) | 89 % |
| Condition MeSH terms filled | 70 % – missing mainly in healthy-volunteer trials (condition "Healthy") → fallback on free-text conditions needed in notebook 02 |

The key numbers are also recorded in `docs/project_log.md`.

## Part B – EU Clinical Trials Register (EudraCT)

The EU Clinical Trials Register (EU CTR) has no API. Its search page offers a plain-text
download of **"Full Trial Details" for the 20 trials shown on the current page**
(`/ctr-search/rest/download/full?<query>&page=N&mode=current_page`). We therefore download
page by page:

* **Query:** `phase=phase-one` (EudraCT field E.7.1 "Human pharmacology (Phase I)" = Yes;
  combined Phase I/II trials are included).
* **Polite crawling:** a pause between requests, retries with increasing waiting time.
* **Caching:** every page is saved as its own text file; pages already on disk are not
  downloaded again.
* **Check:** each page must contain 20 trials (last page: the remainder), and the total
  number of unique EudraCT numbers must equal the count shown on the search page.

One downloaded record = one trial in **one member state** (multi-state trials have one
record per country). Records are combined per EudraCT number later.

In [16]:
import time

EUCTR_BASE = "https://www.clinicaltrialsregister.eu/ctr-search"
EUCTR_QUERY = "query=&phase=phase-one"
EUCTR_PAGE_DIR = PATHS["raw_eudract"] / "pages"
EUCTR_PAGE_DIR.mkdir(parents=True, exist_ok=True)
EUCTR_META_FILE = PATHS["raw_eudract"] / "euctr_phase1_metadata.json"
PAGE_SIZE = 20
PAUSE_SECONDS = 1.5

session = requests.Session()
session.headers["User-Agent"] = "phase1-terminations-europe (research project; polite crawler)"


def get_with_retries(url, retries=4, timeout=180):
    """GET with retries and exponential backoff (2, 4, 8, 16 s)."""
    for attempt in range(retries + 1):
        try:
            r = session.get(url, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.RequestException as err:
            if attempt == retries:
                raise
            wait = 2 ** (attempt + 1)
            print(f"    retry in {wait}s ({err.__class__.__name__})")
            time.sleep(wait)


def euctr_result_count(query):
    """Number of trials shown on the search page (tab 'Trials with a EudraCT protocol')."""
    html = get_with_retries(f"{EUCTR_BASE}/search?{query}").text
    return int(re.search(r"([\d,]+) result\(s\) found", html).group(1).replace(",", ""))


if OFFLINE and EUCTR_META_FILE.exists():     # re-run: counts as recorded at download time
    _m = json.loads(EUCTR_META_FILE.read_text())
    n_trials_search, n_trials_with_results = _m["n_trials_search_page"], _m["n_trials_with_results_search_page"]
    print(f"OFFLINE: counts taken from {EUCTR_META_FILE.name} (downloaded {_m['downloaded_at_utc'][:10]})")
else:
    n_trials_search = euctr_result_count(EUCTR_QUERY)
    n_trials_with_results = euctr_result_count(EUCTR_QUERY + "&resultsstatus=trials-with-results")
n_pages = -(-n_trials_search // PAGE_SIZE)   # ceiling division
print(f"Phase I trials on search page : {n_trials_search:,}  ({n_pages} pages)")
print(f"  ... of which with results   : {n_trials_with_results:,}")

OFFLINE: counts taken from euctr_phase1_metadata.json (downloaded 2026-10-27)
Phase I trials on search page : 2,823  (142 pages)
  ... of which with results   : 1,658


In [17]:
def page_file(page):
    return EUCTR_PAGE_DIR / f"page_{page:03d}.txt"


def trials_on_page(text):
    return re.findall(r"^EudraCT Number: (\S+)", text, re.M)


for page in range(1, n_pages + 1):
    expected = PAGE_SIZE if page < n_pages else n_trials_search - PAGE_SIZE * (n_pages - 1)
    if page_file(page).exists():
        continue
    url = f"{EUCTR_BASE}/rest/download/full?{EUCTR_QUERY}&page={page}&mode=current_page"
    text = get_with_retries(url).text
    n_found = len(set(trials_on_page(text)))
    if n_found != expected:
        print(f"  page {page}: expected {expected} trials, found {n_found}")
    page_file(page).write_text(text, encoding="utf-8")
    if page % 10 == 0:
        print(f"  downloaded page {page} / {n_pages}")
    time.sleep(PAUSE_SECONDS)

euctr_texts = [page_file(p).read_text(encoding="utf-8") for p in range(1, n_pages + 1)]
all_ids = [e for t in euctr_texts for e in trials_on_page(t)]
print(f"Pages on disk                 : {len(euctr_texts)}")
print(f"Country records downloaded    : {len(all_ids):,}")
print(f"Unique EudraCT numbers        : {len(set(all_ids)):,} (search page: {n_trials_search:,})")

if not EUCTR_META_FILE.exists():
    EUCTR_META_FILE.write_text(json.dumps({
        "source": "EU Clinical Trials Register (clinicaltrialsregister.eu)",
        "download_endpoint": f"{EUCTR_BASE}/rest/download/full",
        "query": EUCTR_QUERY,
        "content": "Full Trial Details, plain text, 20 trials per page (mode=current_page)",
        "n_trials_search_page": n_trials_search,
        "n_trials_with_results_search_page": n_trials_with_results,
        "n_pages": n_pages,
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }, indent=2))
print(json.loads(EUCTR_META_FILE.read_text()))

Pages on disk                 : 142
Country records downloaded    : 6,083
Unique EudraCT numbers        : 2,823 (search page: 2,823)
{'source': 'EU Clinical Trials Register (clinicaltrialsregister.eu)', 'download_endpoint': 'https://www.clinicaltrialsregister.eu/ctr-search/rest/download/full', 'query': 'query=&phase=phase-one', 'content': 'Full Trial Details, plain text, 20 trials per page (mode=current_page)', 'n_trials_search_page': 2823, 'n_trials_with_results_search_page': 1658, 'n_pages': 142, 'downloaded_at_utc': '2026-10-27T21:19:18+00:00'}


### B2. Parse the text export

Each record starts with a line `Summary`. Fields are lines of the form `label: value`; we
extract the fields needed for the transparency comparison and the cross-check with
ClinicalTrials.gov (first occurrence per record, e.g. the first sponsor). Records of trials
run **outside the EU/EEA** (paediatric trials listed because they belong to a PIP) have no
member-state section and no trial status.

In [18]:
EUCTR_FIELDS = {
    # column name: exact label in the text export (first occurrence per record is used)
    "eudract_number": "EudraCT Number",
    "national_competent_authority": "National Competent Authority",
    "clinical_trial_type": "Clinical Trial Type",
    "trial_status": "Trial Status",
    "first_entered_date": "Date on which this record was first entered in the EudraCT database",
    "link": "Link",
    "member_state": "A.1 Member State Concerned",
    "full_title": "A.3 Full title of the trial",
    "part_of_pip": "A.7 Trial is part of a Paediatric Investigation Plan",
    "sponsor_name": "B.1.1 Name of Sponsor",
    "sponsor_country": "B.1.3.4\tCountry",
    "sponsor_status": "B.3.1 and B.3.2\tStatus of the sponsor",
    "condition": "E.1.1 Medical condition(s) being investigated",
    "therapeutic_area": "E.1.1.2 Therapeutic area",
    "phase_1": "E.7.1 Human pharmacology (Phase I)",
    "first_in_human": "E.7.1.1 First administration to humans",
    "bioequivalence": "E.7.1.2 Bioequivalence study",
    "phase_2": "E.7.2 Therapeutic exploratory (Phase II)",
    "phase_3": "E.7.3 Therapeutic confirmatory (Phase III)",
    "phase_4": "E.7.4 Therapeutic use (Phase IV)",
    "randomised": "E.8.1.1 Randomised",
    "double_blind": "E.8.1.4 Double blind",
    "placebo_comparator": "E.8.2.2 Placebo",
    "multiple_member_states": "E.8.5 The trial involves multiple Member States",
    "has_dmc": "E.8.7 Trial has a data monitoring committee",
    "subjects_under_18": "F.1.1 Trial has subjects under 18",
    "adults_18_64": "F.1.2 Adults (18-64 years)",
    "elderly_65plus": "F.1.3 Elderly (>=65 years)",
    "healthy_volunteers": "F.3.1 Healthy volunteers",
    "patients": "F.3.2 Patients",
    "planned_subjects_member_state": "F.4.1 In the member state",
    "planned_subjects_whole_trial": "F.4.2.2 In the whole clinical trial",
    "ca_decision": "N. Competent Authority Decision",
    "ca_decision_date": "N. Date of Competent Authority Decision",
    "end_of_trial_status": "P. End of Trial Status",
    "global_end_date": "P. Date of the global end of the trial",
}
YES_NO_COLUMNS = ["part_of_pip", "phase_1", "first_in_human", "bioequivalence", "phase_2",
                  "phase_3", "phase_4", "randomised", "double_blind", "placebo_comparator",
                  "multiple_member_states", "has_dmc", "subjects_under_18", "adults_18_64",
                  "elderly_65plus", "healthy_volunteers", "patients"]
FIELD_PATTERNS = {col: re.compile(rf"^{re.escape(label)}:[ \t]*(.*?)[ \t]*$", re.M)
                  for col, label in EUCTR_FIELDS.items()}


def parse_euctr_text(text):
    """Split a downloaded page into records (one per trial and member state) and extract fields."""
    records = []
    for block in re.split(r"^Summary$", text, flags=re.M)[1:]:
        rec = {}
        for col, pattern in FIELD_PATTERNS.items():
            m = pattern.search(block)
            rec[col] = (m.group(1).strip() or None) if m else None
        records.append(rec)
    return records

euctr_records = pd.DataFrame([r for t in euctr_texts for r in parse_euctr_text(t)])
for col in YES_NO_COLUMNS:
    euctr_records[col] = euctr_records[col].map({"Yes": True, "No": False})
print(euctr_records.shape)
(euctr_records.notna().mean() * 100).round(1).rename("% non-missing").to_frame().T

(6083, 36)


,eudract_number,national_competent_authority,clinical_trial_type,trial_status,first_entered_date,link,member_state,full_title,part_of_pip,sponsor_name,sponsor_country,sponsor_status,condition,therapeutic_area,phase_1,first_in_human,bioequivalence,phase_2,phase_3,phase_4,randomised,double_blind,placebo_comparator,multiple_member_states,has_dmc,subjects_under_18,adults_18_64,elderly_65plus,healthy_volunteers,patients,planned_subjects_member_state,planned_subjects_whole_trial,ca_decision,ca_decision_date,end_of_trial_status,global_end_date
% non-missing,100.0,95.8,100.0,94.7,100.0,100.0,95.8,99.9,85.3,99.9,99.7,99.8,99.5,85.1,100.0,98.7,98.5,99.7,98.8,98.8,93.1,90.9,86.8,95.4,97.6,100.0,99.8,99.5,99.8,99.9,93.8,86.9,95.8,95.8,94.7,49.4


In [19]:
print(euctr_records["clinical_trial_type"].value_counts(dropna=False), "\n")
print(euctr_records["trial_status"].value_counts(dropna=False))

clinical_trial_type
EEA CTA           5829
Outside EU/EEA     254
Name: count, dtype: int64 

trial_status
Completed                   3145
Prematurely Ended            945
Trial now transitioned       897
Ongoing                      433
NaN                          321
GB - no longer in EU/EEA     284
Temporarily Halted            39
Restarted                      9
Not Authorised                 8
Suspended by CA                1
Prohibited by CA               1
Name: count, dtype: int64


### B3. One row per trial (EudraCT number)

Multi-state trials have one record per member state, and the status can differ between
countries. We combine them with a documented priority rule (decision **D21**):

1. any country **Prematurely Ended** → `Prematurely Ended`
2. else any country **Temporarily Halted / Suspended by CA** → `Temporarily Halted`
3. else any country **Ongoing / Restarted** → `Ongoing`
4. else any country **Trial now transitioned** → `Transitioned to CTIS` (trial was still
   running when the transition period ended on 30 January 2025; followed up in CTIS)
5. else any country **Completed** → `Completed`
6. else only **GB – no longer in EU/EEA** → status not visible (UK records after Brexit)
7. else **Not Authorised / Prohibited by CA** → `Not authorised`
8. EEA records with an empty status field → `Not reported`
9. no EEA record at all → `Outside EU/EEA (no status)`

`status_mixed` flags trials whose countries report different statuses. Note: "Prematurely
Ended" in one country can mean that only this country stopped early.

In [20]:
STATUS_PRIORITY = [
    ("Prematurely Ended", {"Prematurely Ended"}),
    ("Temporarily Halted", {"Temporarily Halted", "Suspended by CA"}),
    ("Ongoing", {"Ongoing", "Restarted"}),
    ("Transitioned to CTIS", {"Trial now transitioned"}),
    ("Completed", {"Completed"}),
    ("GB - no longer in EU/EEA", {"GB - no longer in EU/EEA"}),
    ("Not authorised", {"Not Authorised", "Prohibited by CA"}),
]


def combine_status(group):
    present = {s for s in group["trial_status"] if isinstance(s, str)}
    for label, members in STATUS_PRIORITY:
        if present & members:
            return label
    if present:
        return "Other: " + "|".join(sorted(present))
    if (group["clinical_trial_type"] == "EEA CTA").any():
        return "Not reported"
    return "Outside EU/EEA (no status)"


def any_true(s):
    return bool(s.fillna(False).any())


grouped = euctr_records.groupby("eudract_number", sort=False)
euctr_trials = grouped.agg(
    n_records=("eudract_number", "size"),
    member_states=("member_state", lambda s: "|".join(sorted(s.dropna().unique())) or None),
    statuses=("trial_status", lambda s: "|".join(sorted(s.dropna().unique())) or None),
    first_entered_date=("first_entered_date", "min"),
    full_title=("full_title", "first"),
    sponsor_name=("sponsor_name", "first"),
    sponsor_status=("sponsor_status", "first"),
    therapeutic_area=("therapeutic_area", "first"),
    **{col: (col, any_true) for col in YES_NO_COLUMNS},
).reset_index()
euctr_trials["overall_status"] = grouped[["trial_status", "clinical_trial_type"]].apply(combine_status).values
euctr_trials["status_mixed"] = euctr_trials["statuses"].fillna("").str.count(r"\|") > 0
euctr_trials["only_outside_eea"] = grouped["clinical_trial_type"].apply(
    lambda s: set(s.dropna()) == {"Outside EU/EEA"}).values

print(f"Country records : {len(euctr_records):,}")
print(f"Unique trials   : {len(euctr_trials):,} (search page: {n_trials_search:,})")
assert len(euctr_trials) == n_trials_search, "Number of trials differs from search page"
print(euctr_trials["overall_status"].value_counts(dropna=False).to_string())
print(f"\nTrials with mixed statuses across countries: {euctr_trials['status_mixed'].sum():,}")

Country records : 6,083
Unique trials   : 2,823 (search page: 2,823)
overall_status
Completed                     1264
Prematurely Ended              591
Ongoing                        370
Transitioned to CTIS           264
Outside EU/EEA (no status)     195
GB - no longer in EU/EEA        89
Temporarily Halted              30
Not reported                    17
Not authorised                   3

Trials with mixed statuses across countries: 583


### B4. Which Phase I trials are public in EudraCT?

Under Directive 2001/20/EC, protocol information of Phase I trials in adults was not
public; exceptions are trials that are part of a Paediatric Investigation Plan (PIP) or
include children. We check what the downloaded trials actually are.

In [21]:
t = euctr_trials
phase_combo = (t["phase_1"].map({True: "I", False: ""})
               + t["phase_2"].map({True: "+II", False: ""})
               + t["phase_3"].map({True: "+III", False: ""})
               + t["phase_4"].map({True: "+IV", False: ""}))
visibility_group = pd.Series("Adults only, not PIP", index=t.index)
visibility_group[t["subjects_under_18"]] = "Includes children (<18)"
visibility_group[t["part_of_pip"]] = "Part of a PIP"
print(pd.crosstab(visibility_group, phase_combo, margins=True), "\n")
print("Healthy volunteers:", t["healthy_volunteers"].sum(), "| first-in-human:", t["first_in_human"].sum())

col_0                      I  I+II  I+II+III  I+II+III+IV  I+II+IV  I+III  I+III+IV  I+IV   All
row_0                                                                                          
Adults only, not PIP       4  1761        20            1        0     38         3    48  1875
Includes children (<18)  317   232        11            0        2      8         2     8   580
Part of a PIP            244   107        10            0        0      7         0     0   368
All                      565  2100        41            1        2     53         5    56  2823 

Healthy volunteers: 311 | first-in-human: 833


### B5. Save

Country-level records and trial-level table are saved to `data/interim/`.

In [22]:
euctr_records.to_parquet(PATHS["interim"] / "euctr_records.parquet", index=False)
euctr_trials.to_parquet(PATHS["interim"] / "euctr_trials.parquet", index=False)
print("Saved:", len(euctr_records), "records,", len(euctr_trials), "trials")

Saved: 6083 records, 2823 trials


### B6. Summary of the EudraCT download

Download: 2026-10-27, query `phase=phase-one`, 142 pages, "Full Trial Details".

| Check | Result |
|---|---|
| Trials on search page vs. unique EudraCT numbers downloaded | 2,823 / 2,823 – complete |
| Country records (one per member state) | 6,083 (5,829 EEA, 254 outside EU/EEA) |
| Trials with results (search filter) | 1,658 (59 %) |
| Trials with different statuses across countries | 583 (21 %) |
| **Adult, non-PIP trials that are Phase I only** | **4** – vs. 1,761 adult non-PIP trials that are Phase I/II |
| Phase I-only trials visible | 565, of which 561 include children or are part of a PIP |
| Healthy-volunteer trials | 311 |

**Interpretation:** the register confirms the transparency gap of Directive 2001/20/EC –
pure Phase I trials in adults are practically invisible; what is visible are paediatric/PIP
trials and combined Phase I/II trials. In addition, UK records show only "GB – no longer in
EU/EEA" instead of their real status, and 264 trials have been transitioned to CTIS.

The EU CTR export contains **no reason for premature termination** in the protocol data.

## Part C – CTIS (euclinicaltrials.eu)

Data are retrieved via the portal's **public search interface** (not a documented API; it
may change over time – download date recorded). Terms of use: EMA Legal Notice, see
`docs/data_sources_and_terms.md`.

* `POST /ctis-public-api/search` – list of trials with basic fields (page size up to 500).
  The interface returns **at most 10,000 records per query**; we therefore query twice
  (sorted ascending and descending by trial number) and combine both lists.
* `GET /ctis-public-api/retrieve/{ctNumber}` – full public record of one trial, including
  member-state events (start, end, **early termination with a coded reason**, temporary
  halts) and whether the trial was **transitioned** from EudraCT.

Phase I trials are selected by the phase text shown in the portal: "Human Pharmacology
(Phase I)" and "Phase I and Phase II (Integrated)". Numeric status codes are translated with
the label list used by the portal itself.

In [23]:
CTIS_API = "https://euclinicaltrials.eu/ctis-public-api"
CTIS_SEARCH_FILE = PATHS["raw_ctis"] / "ctis_search_all_raw.json.gz"
CTIS_DETAILS_FILE = PATHS["raw_ctis"] / "ctis_phase1_details_raw.jsonl.gz"
CTIS_META_FILE = PATHS["raw_ctis"] / "ctis_metadata.json"
CTIS_MAX_PER_QUERY = 10_000

# Overall trial status codes as defined in the CTIS public portal
CTIS_STATUS_LABELS = {
    1: "Under evaluation", 2: "Authorised, recruitment pending", 3: "Authorised, recruiting",
    4: "Ongoing, recruiting", 5: "Ongoing, recruitment ended", 6: "Temporarily halted",
    7: "Suspended", 8: "Ended", 9: "Expired", 10: "Revoked", 11: "Not authorised",
    12: "Cancelled",
}


def ctis_search_page(page, direction, size=500):
    body = {"pagination": {"page": page, "size": size},
            "sort": {"property": "ctNumber", "direction": direction},
            "searchCriteria": {}}
    for attempt in range(5):
        try:
            r = session.post(f"{CTIS_API}/search", json=body, timeout=180)
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if attempt == 4:
                raise
            time.sleep(2 ** (attempt + 1))


if CTIS_SEARCH_FILE.exists():
    with gzip.open(CTIS_SEARCH_FILE, "rt", encoding="utf-8") as f:
        saved = json.load(f)
    ctis_search, ctis_total = saved["records"], saved["total_records_reported"]
    print(f"Loaded {len(ctis_search):,} search records from saved raw file")
else:
    by_number = {}
    for direction in ("ASC", "DESC"):
        page, n_dir = 1, 0
        while n_dir < CTIS_MAX_PER_QUERY:
            data = ctis_search_page(page, direction)
            ctis_total = data["pagination"]["totalRecords"]
            if not data["data"]:
                break
            for rec in data["data"]:
                by_number[rec["ctNumber"]] = rec
            n_dir += len(data["data"])
            page += 1
            time.sleep(0.5)
        print(f"  {direction}: {n_dir:,} records, combined unique: {len(by_number):,}")
    ctis_search = list(by_number.values())
    with gzip.open(CTIS_SEARCH_FILE, "wt", encoding="utf-8") as f:
        json.dump({"total_records_reported": ctis_total, "records": ctis_search}, f)

print(f"CTIS trials reported: {ctis_total:,} | unique trials downloaded: {len(ctis_search):,}")
assert len(ctis_search) == ctis_total, "CTIS search download incomplete"

Loaded 12,725 search records from saved raw file
CTIS trials reported: 12,725 | unique trials downloaded: 12,725


In [24]:
search_df = pd.DataFrame(ctis_search)
print(search_df["trialPhase"].value_counts(dropna=False).to_string())

is_phase1 = (search_df["trialPhase"].fillna("").str.contains("Phase I)", regex=False)
             | search_df["trialPhase"].fillna("").str.startswith("Phase I and Phase II"))
ctis_phase1_numbers = search_df.loc[is_phase1, "ctNumber"].tolist()
print(f"\nPhase I trials (incl. integrated Phase I/II): {len(ctis_phase1_numbers):,}")

trialPhase
Therapeutic exploratory (Phase II)                                   3727
Therapeutic confirmatory  (Phase III)                                3534
Therapeutic use (Phase IV)                                           1292
Human Pharmacology (Phase I)-  Other                                 1154
Human Pharmacology (Phase I)- First administration to humans          876
Phase I and Phase II (Integrated)- Other                              670
Phase I and Phase II (Integrated)- First administration to humans     485
Phase II and Phase III (Integrated)                                   418
Human Pharmacology (Phase I)- Bioequivalence Study                    324
Phase III and phase IV (Integrated)                                   232
Phase I and Phase II (Integrated)- Bioequivalence Study                13

Phase I trials (incl. integrated Phase I/II): 3,522


In [25]:
def load_details():
    details = {}
    if CTIS_DETAILS_FILE.exists():
        with gzip.open(CTIS_DETAILS_FILE, "rt", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                details[rec["ctNumber"]] = rec
    return details


ctis_details = load_details()
missing = [n for n in ctis_phase1_numbers if n not in ctis_details]
print(f"Details on disk: {len(ctis_details):,} | still to download: {len(missing):,}")

with gzip.open(CTIS_DETAILS_FILE, "at", encoding="utf-8") as f:   # append = resumable
    for i, number in enumerate(missing, 1):
        rec = get_with_retries(f"{CTIS_API}/retrieve/{number}").json()
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        ctis_details[number] = rec
        if i % 250 == 0:
            f.flush()
            print(f"  downloaded {i:,} / {len(missing):,}")
        time.sleep(0.3)

print(f"Details available: {len(ctis_details):,} / {len(ctis_phase1_numbers):,}")
assert set(ctis_phase1_numbers) <= set(ctis_details), "CTIS details incomplete"

if not CTIS_META_FILE.exists():
    CTIS_META_FILE.write_text(json.dumps({
        "source": "CTIS public portal (euclinicaltrials.eu)",
        "search_endpoint": f"{CTIS_API}/search (POST, empty search criteria, sorted ASC and DESC)",
        "details_endpoint": f"{CTIS_API}/retrieve/{{ctNumber}}",
        "phase_selection": "trialPhase contains 'Phase I)' or starts with 'Phase I and Phase II'",
        "n_trials_total": ctis_total,
        "n_phase1_trials": len(ctis_phase1_numbers),
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }, indent=2))
print(json.loads(CTIS_META_FILE.read_text()))

Details on disk: 3,522 | still to download: 0
Details available: 3,522 / 3,522
{'source': 'CTIS public portal (euclinicaltrials.eu)', 'search_endpoint': 'https://euclinicaltrials.eu/ctis-public-api/search (POST, empty search criteria, sorted ASC and DESC)', 'details_endpoint': 'https://euclinicaltrials.eu/ctis-public-api/retrieve/{ctNumber}', 'phase_selection': "trialPhase contains 'Phase I)' or starts with 'Phase I and Phase II'", 'n_trials_total': 12725, 'n_phase1_trials': 3522, 'downloaded_at_utc': '2026-10-28T14:46:16+00:00'}


### C2. Flatten to one row per trial

Fields come from the search list (phase, countries with number of sites, sponsor type, age
group) and from the full record (population, transition from EudraCT, member-state events,
results). For **early termination**, CTIS records a member-state event `EARLY_TERMINATION`
with a **coded reason** – a structured field that neither EudraCT nor ClinicalTrials.gov has.

* `early_termination_any`: at least one member state reported an early termination
  (comparable to the EudraCT priority rule D21)
* `early_termination_all`: every member state with events reported an early termination
  (strict variant, comparable to D22)

In [26]:
TARGET_COUNTRIES_EU = [c for c in TARGET_COUNTRIES if c != "United Kingdom"]   # UK is not in CTIS


def flatten_ctis(search_rec, detail):
    aa = detail.get("authorizedApplication") or {}
    part1 = aa.get("authorizedPartI") or {}
    trial_info = get(part1, "trialDetails", "trialInformation", default={}) or {}
    events = detail.get("events") or {}
    msc_events = events.get("trialEvents") or []

    # Member-state events: early termination and its coded reason
    early = [m for m in msc_events
             if any(e.get("notificationType") == "EARLY_TERMINATION" for e in m.get("events", []))]
    ended = [m for m in msc_events
             if any(e.get("notificationType") in ("END_OF_TRIAL", "EARLY_TERMINATION")
                    for e in m.get("events", []))]
    reasons = sorted({get(m, "earlyTerminationReason", "name") for m in early} - {None})

    countries = [c.split(":")[0] for c in (search_rec.get("trialCountries") or [])]
    n_sites = sum(int(c.split(":")[1]) for c in (search_rec.get("trialCountries") or [])
                  if ":" in c and c.split(":")[1].isdigit())
    groups = [g.get("name") for g in get(trial_info, "populationOfTrialSubjects",
                                         "clinicalTrialGroups", default=[]) or []]
    results = detail.get("results") or {}
    pip = get(part1, "trialDetails", "scientificAdviceAndPip", "paediatricInvestigationPlan", default=[])
    phase_text = search_rec.get("trialPhase") or ""
    status_code = search_rec.get("ctStatus")

    return {
        "ct_number": search_rec["ctNumber"],
        "title": search_rec.get("ctTitle"),
        "status_code": status_code,
        "status_label": CTIS_STATUS_LABELS.get(status_code),
        "trial_phase": phase_text,
        "phase_group": "Phase I/II" if phase_text.startswith("Phase I and Phase II") else "Phase I",
        "first_in_human": "First administration to humans" in phase_text,
        "bioequivalence": "Bioequivalence" in phase_text,
        "trial_category": get(trial_info, "trialCategory", "trialCategory"),
        "decision_date": (detail.get("decisionDate") or "")[:10] or None,
        "sponsor_type": search_rec.get("sponsorType"),
        "conditions": search_rec.get("conditions"),
        "therapeutic_areas": join(search_rec.get("therapeuticAreas") or []),
        "age_group": search_rec.get("ageGroup"),
        "healthy_volunteers": "Healthy volunteers" in groups,
        "patients": "Patients" in groups,
        "population_reported": bool(groups),
        "part_of_pip": bool(pip),
        "countries": join(countries),
        "n_countries": len(countries),
        "n_sites": n_sites,
        "target_countries": join([c for c in countries if c in TARGET_COUNTRIES_EU]),
        "is_transitioned": bool(get(aa, "eudraCt", "isTransitioned")),
        "eudract_number": get(aa, "eudraCt", "eudraCtCode"),
        # events
        "n_msc_with_events": len(msc_events),
        "n_msc_ended": len(ended),
        "n_msc_early_terminated": len(early),
        "early_termination_any": bool(early),
        "early_termination_all": bool(early) and len(early) == len(ended) == len(msc_events),
        "early_termination_reasons": join(reasons),
        "global_end_date": events.get("trialGlobalEndDate"),
        "n_temporary_halts": len(events.get("temporaryHaltList") or []),
        "n_urgent_safety_measures": len(events.get("urgentSafetyMeasures") or []),
        "n_serious_breaches": len(events.get("seriousBreaches") or []),
        "n_unexpected_events": len(events.get("unexpectedEvents") or []),
        "n_corrective_measures": len(detail.get("correctiveMeasures") or []),
        # results
        "results_first_received": search_rec.get("resultsFirstReceived"),
        "has_summary_results": bool(results.get("summaryResults")),
        "has_layperson_results": bool(results.get("laypersonResults")),
    }


ctis_search_by_number = {r["ctNumber"]: r for r in ctis_search}
ctis = pd.DataFrame([flatten_ctis(ctis_search_by_number[n], ctis_details[n])
                     for n in ctis_phase1_numbers])
print(ctis.shape)
ctis.head(3)

(3522, 39)


,ct_number,title,status_code,status_label,trial_phase,phase_group,first_in_human,bioequivalence,trial_category,decision_date,sponsor_type,conditions,therapeutic_areas,age_group,healthy_volunteers,patients,population_reported,part_of_pip,countries,n_countries,n_sites,target_countries,is_transitioned,eudract_number,n_msc_with_events,n_msc_ended,n_msc_early_terminated,early_termination_any,early_termination_all,early_termination_reasons,global_end_date,n_temporary_halts,n_urgent_safety_measures,n_serious_breaches,n_unexpected_events,n_corrective_measures,results_first_received,has_summary_results,has_layperson_results
0,2022-500007-52-00,A study comparing exposure of semaglutide and ...,8,Ended,Human Pharmacology (Phase I)- Bioequivalence S...,Phase I,False,True,1,2022-06-02,Pharmaceutical company,"Diabetes Mellitus, Type 2",Diseases [C] - Nutritional and Metabolic Disea...,18-64 years,True,False,True,False,Germany,1,8,Germany,False,NaN,1,1,0,False,False,NaN,NaN,0,0,0,0,0,Yes,True,True
1,2022-500041-24-00,SPRING Rollover,4,"Ongoing, recruiting",Phase I and Phase II (Integrated)- Other,Phase I/II,False,False,1,2022-06-01,Non-Pharmaceutical company,Lung cancer (non-small cell lung cancer),Diseases [C] - Neoplasms [C04],65+ years,False,True,True,False,Spain,1,4,Spain,False,NaN,1,1,0,False,False,NaN,NaN,0,0,0,0,0,No,False,False
2,2022-500050-42-00,The effect of multiple doses of BI 425809 on t...,8,Ended,Human Pharmacology (Phase I)- Other,Phase I,False,False,1,2022-11-10,Pharmaceutical company,Healthy,Phenomena and Processes [G] - Reproductive and...,18-64 years,True,False,True,False,Germany,1,8,Germany,False,NaN,1,1,0,False,False,NaN,NaN,0,0,0,0,0,Yes,True,True


### C3. Status mapping CTIS → ClinicalTrials.gov

The mapping table `mappings/ctis_status_mapping.csv` assigns each CTIS overall status an
equivalent ClinicalTrials.gov status. "Ended" is split with the member-state events:
**Ended + early termination in any member state → TERMINATED**, otherwise COMPLETED.

In [27]:
status_map = pd.read_csv(PROJECT_DIR / "mappings" / "ctis_status_mapping.csv")
ctis = ctis.merge(status_map[["ctis_code", "ctgov_equivalent", "status_group"]],
                  left_on="status_code", right_on="ctis_code", how="left").drop(columns="ctis_code")
ctis["status_ctgov_like"] = ctis["ctgov_equivalent"]
ended = ctis["status_code"] == 8
ctis.loc[ended, "status_ctgov_like"] = ctis.loc[ended, "early_termination_any"].map(
    {True: "TERMINATED", False: "COMPLETED"})
ctis = ctis.drop(columns="ctgov_equivalent")
assert ctis["status_group"].notna().all(), "Unmapped CTIS status code"

print(pd.crosstab(ctis["status_label"], ctis["status_ctgov_like"]).to_string())

status_ctgov_like                ACTIVE_NOT_RECRUITING  COMPLETED  NOT_YET_RECRUITING  RECRUITING  SUSPENDED  TERMINATED  WITHDRAWN
status_label                                                                                                                       
Authorised, recruiting                               0          0                   0         180          0           0          0
Authorised, recruitment pending                      0          0                 304           0          0           0          0
Ended                                                0       1162                   0           0          0         441          0
Expired                                              0          0                   0           0          0           0          3
Not authorised                                       0          0                   0           0          0           0         60
Ongoing, recruiting                                  0          0           

### C4. Input checks

In [28]:
print(f"Phase I trials            : {len(ctis):,}")
print(f"Duplicate CT numbers      : {ctis['ct_number'].duplicated().sum()}")
print(f"Phase group               : {ctis['phase_group'].value_counts().to_dict()}")
print(f"First-in-human            : {ctis['first_in_human'].sum():,}")
print(f"Decision year             : {pd.to_datetime(ctis['decision_date']).dt.year.value_counts().sort_index().to_dict()}")
print(f"Trial category (1 = Phase I category of the transparency rules): "
      f"{ctis['trial_category'].value_counts(dropna=False).to_dict()}")
print(f"Site in DE/FR/ES/NL/BE/IT : {ctis['target_countries'].notna().sum():,}")
print(f"Healthy volunteers        : {ctis['healthy_volunteers'].sum():,} "
      f"({100 * ctis['healthy_volunteers'].mean():.0f} %)")
print(f"Part of a PIP             : {ctis['part_of_pip'].sum():,}")

Phase I trials            : 3,522
Duplicate CT numbers      : 0
Phase group               : {'Phase I': 2354, 'Phase I/II': 1168}
First-in-human            : 1,361
Decision year             : {2022: 52, 2023: 528, 2024: 1595, 2025: 721, 2026: 626}
Trial category (1 = Phase I category of the transparency rules): {'1': 2463, '2': 1048, '3': 11}
Site in DE/FR/ES/NL/BE/IT : 2,868
Healthy volunteers        : 1,210 (34 %)
Part of a PIP             : 98


In [29]:
ended_df = ctis[ctis["status_code"] == 8]
print(f"Ended trials                          : {len(ended_df):,}")
print(f"  early termination in >= 1 country   : {ended_df['early_termination_any'].sum():,} "
      f"({100 * ended_df['early_termination_any'].mean():.1f} %)")
print(f"  early termination in all countries  : {ended_df['early_termination_all'].sum():,} "
      f"({100 * ended_df['early_termination_all'].mean():.1f} %)")
print("\nCoded early-termination reasons (trials; a trial can have several):")
print(ctis["early_termination_reasons"].dropna().str.split("|").explode().value_counts().to_string())

Ended trials                          : 1,603
  early termination in >= 1 country   : 441 (27.5 %)
  early termination in all countries  : 360 (22.5 %)

Coded early-termination reasons (trials; a trial can have several):
early_termination_reasons
Sponsor Decision                                     429
Feasibility (recruitment issues, etc)                 57
Safety Related (Clinical or Pre-Clinical Results)     17
Study Related                                         15
Investigator / Site related                           13
Reprioritisation of Trial                             10
Medicinal Product Related                              4
Merger                                                 1
Divestment                                             1


In [30]:
# Safety-related events and results availability (transparency indicators)
indicators = {
    "Temporary halt notified": (ctis["n_temporary_halts"] > 0).sum(),
    "Urgent safety measure": (ctis["n_urgent_safety_measures"] > 0).sum(),
    "Unexpected event": (ctis["n_unexpected_events"] > 0).sum(),
    "Serious breach": (ctis["n_serious_breaches"] > 0).sum(),
    "Corrective measure": (ctis["n_corrective_measures"] > 0).sum(),
    "Summary results (all trials)": ctis["has_summary_results"].sum(),
    "Summary results (ended trials)": ended_df["has_summary_results"].sum(),
    "Layperson summary (ended trials)": ended_df["has_layperson_results"].sum(),
}
pd.Series(indicators, name="trials").to_frame()

,trials
Temporary halt notified,217
Urgent safety measure,28
Unexpected event,28
Serious breach,24
Corrective measure,45
Summary results (all trials),326
Summary results (ended trials),325
Layperson summary (ended trials),324


#### Transition from EudraCT

Trials still running on 30 January 2025 had to be transitioned to CTIS. They keep a link to
their EudraCT number, which we compare with the EudraCT download (Part B).

In [31]:
trans = ctis[ctis["is_transitioned"]]
in_euctr = trans["eudract_number"].isin(euctr_trials["eudract_number"])
print(f"Transitioned Phase I trials in CTIS          : {len(trans):,}")
print(f"  ... with EudraCT number                    : {trans['eudract_number'].notna().sum():,}")
print(f"  ... found in EU CTR Phase I download       : {in_euctr.sum():,}")
print(f"EU CTR trials with status 'Transitioned'     : "
      f"{(euctr_trials['overall_status'] == 'Transitioned to CTIS').sum():,}")

Transitioned Phase I trials in CTIS          : 952
  ... with EudraCT number                    : 952
  ... found in EU CTR Phase I download       : 424
EU CTR trials with status 'Transitioned'     : 264


### C5. Save

In [32]:
ctis.to_parquet(PATHS["interim"] / "ctis_phase1.parquet", index=False)
print(f"Saved {len(ctis):,} rows x {ctis.shape[1]} columns")

Saved 3,522 rows x 41 columns


### C6. Summary of the CTIS download

Download: 2026-10-28; all 12,725 public CTIS trials (search list), full records for the
3,522 Phase I trials (incl. integrated Phase I/II).

| Check | Result |
|---|---|
| Search list vs. reported total | 12,725 / 12,725 – complete (10,000-record limit bypassed with two sort orders) |
| Phase I detail records | 3,522 / 3,522; 0 duplicates |
| Phase I / Phase I/II | 2,354 / 1,168 (first-in-human: 1,361) |
| Site in one of the 6 EU target countries | 2,868 (UK is not part of CTIS) |
| Healthy volunteers | 1,210 (34 %) |
| Ended trials | 1,603 – early termination in ≥ 1 country: 441 (27.5 %); in all countries: 360 (22.5 %) |
| Coded termination reason (trials with early termination in ≥ 1 country, any status, n = 536; several codes per trial possible) | Sponsor decision 429 (80.0 %) · Feasibility 57 · Safety 17 · Study-related 15 · Site-related 13 · Reprioritisation 10 · other 6 |
| Transitioned from EudraCT | 952 – of which **528 were not visible** in the EU CTR Phase I download |
| Summary results available (ended trials) | 325 of 1,603 |

**Interpretation (descriptive only):**
* CTIS makes adult Phase I trials public – including 528 transitioned trials that were not
  publicly visible as Phase I trials in the EU CTR.
* CTIS records a **coded** reason for early termination, but "Sponsor decision" is named for
  80 % of early-terminated trials (429 of 536) and is the only code for 79 % of the ended,
  early-terminated trials (notebook 07, Part B), so the reason is formally present but rarely
  informative. *(Corrected in Block 19: an earlier version said 97 % = 429/441, mixing two
  denominators.)*
* Early termination is much more frequent among transitioned trials (43 % vs. 22 %) and among
  patient trials than in healthy-volunteer trials (51 % vs. 8 %) – not comparable with
  ClinicalTrials.gov without adjustment (short observation period, selection at transition).